# BEE Score Analysis: GPT-2 vs Medical-GPT2 on PubMed-200K-RCT

This notebook replicates the BEE (Bias Embedding Evaluation) fingerprinting experiment **without any fine-tuning**:

- **Target model**: `devmanpreet/Medical-GPT2-Classifier` — already fine-tuned on medical text; used as-is.
- **Reference model**: `openai-community/gpt2` — base GPT-2; we fit a sklearn LogisticRegression on its **frozen** embeddings (no PyTorch training).
- **Dataset**: `pietrolesci/pubmed-200k-rct` — PubMed 200K RCT abstracts with section labels.

**Hypothesis**: If the Medical-GPT2-Classifier was trained on data similar to PubMed-200K-RCT, its BEE fingerprint (keyword-class bias scores) should correlate with the BEE fingerprint of a probe trained on PubMed data from scratch.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers datasets torch yake nltk scikit-learn matplotlib seaborn tqdm openpyxl

## 2. Reproducibility — Set All Seeds

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

# Required for CUDA deterministic ops
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True)
    print("Deterministic algorithms: ENABLED")
except Exception as e:
    print(f"Warning — deterministic algorithms not fully available: {e}")

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"SEED = {SEED}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Imports

In [ ]:
import gc
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2Model,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_sim

import nltk
import yake

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 4. Configuration

In [ ]:
# --------------------------------------------------------------------------
# Model identifiers
# --------------------------------------------------------------------------
MODEL_BASE_ID   = "openai-community/gpt2"
MODEL_TARGET_ID = "devmanpreet/Medical-GPT2-Classifier"

# --------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------
DATASET_NAME = "pietrolesci/pubmed-200k-rct"

# --------------------------------------------------------------------------
# Tokenisation
# --------------------------------------------------------------------------
MAX_LENGTH = 128

# --------------------------------------------------------------------------
# Keyword extraction (YAKE)
# --------------------------------------------------------------------------
YAKE_TOP_KEYWORDS = 1000   # keywords to extract
# N_KEYWORD_TEXTS removed — all samples are used

# --------------------------------------------------------------------------
# Sklearn probe
# --------------------------------------------------------------------------
N_PROBE_SAMPLES = 20_000   # samples used to fit the LogisticRegression probe
LR_MAX_ITER     = 2000

# --------------------------------------------------------------------------
# BEE visualisation
# --------------------------------------------------------------------------
TOP_K_DISPLAY   = 15   # keywords shown per class heatmap
TOP_K_OVERLAP   = 20   # keywords used for Jaccard overlap comparison

print("Configuration loaded.")
print(f"  Base model   : {MODEL_BASE_ID}")
print(f"  Target model : {MODEL_TARGET_ID}")
print(f"  Dataset      : {DATASET_NAME}")

## 5. Load Dataset

In [ ]:
print(f"Downloading: {DATASET_NAME}")
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print("\nFeatures:", raw_dataset['train'].features)
print("\nSample row:")
print(raw_dataset['train'][0])

## 6. Dataset Exploration & Preprocessing

In [ ]:
# Combine ALL labeled splits — train + validation + test
split_dfs = []
for split_name in raw_dataset.keys():
    df_split = raw_dataset[split_name].to_pandas()
    df_split['_split'] = split_name
    split_dfs.append(df_split)
    print(f"  {split_name:<12s}: {len(df_split):,} rows")

train_df = pd.concat(split_dfs, ignore_index=True)

print(f"\nCombined shape : {train_df.shape}")
print(f"Columns        : {train_df.columns.tolist()}")

In [ ]:
# --------------------------------------------------------------------------
# Detect text column dynamically
# --------------------------------------------------------------------------
sample = raw_dataset['train'][0]

if 'sentence' in sample:
    TEXT_COL = 'sentence'
elif 'text' in sample:
    TEXT_COL = 'text'
elif 'sentence_text' in sample:
    TEXT_COL = 'sentence_text'
else:
    TEXT_COL = next(
        (k for k, v in sample.items() if isinstance(v, str) and k not in ('label', 'labels', '_split', 'uid')),
        None
    )
    if TEXT_COL is None:
        raise ValueError(f"Cannot detect text column. Keys: {list(sample.keys())}")

# --------------------------------------------------------------------------
# Detect label column dynamically (handles 'label', 'labels', etc.)
# --------------------------------------------------------------------------
for candidate in ('label', 'labels'):
    if candidate in train_df.columns:
        LABEL_COL = candidate
        break
else:
    raise ValueError(f"Cannot find label column. Columns: {train_df.columns.tolist()}")

print(f"Text column  : '{TEXT_COL}'")
print(f"Label column : '{LABEL_COL}'")

# --------------------------------------------------------------------------
# Handle ClassLabel (integer) vs string labels
# --------------------------------------------------------------------------
label_feature = raw_dataset['train'].features[LABEL_COL]
print(f"Label feature type: {type(label_feature).__name__}")

if hasattr(label_feature, 'names'):
    int2str = {i: name for i, name in enumerate(label_feature.names)}
    train_df['label_str'] = train_df[LABEL_COL].map(int2str)
    LABEL_COL = 'label_str'
    print(f"Mapped ClassLabel integers → strings: {int2str}")

# Normalise to upper-case
train_df[LABEL_COL] = train_df[LABEL_COL].astype(str).str.upper().str.strip()

# Drop rows with missing text/label
train_df = train_df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)

# Encode labels
le_pubmed = LabelEncoder()
le_pubmed.fit(sorted(train_df[LABEL_COL].unique()))   # sort for determinism
train_df['label_id'] = le_pubmed.transform(train_df[LABEL_COL])

PUBMED_ID2LABEL = {i: label for i, label in enumerate(le_pubmed.classes_)}
PUBMED_LABEL2ID = {v: k for k, v in PUBMED_ID2LABEL.items()}
PUBMED_NUM_LABELS = len(PUBMED_ID2LABEL)

print(f"\nPubMed label mapping ({PUBMED_NUM_LABELS} classes):")
for idx, label in PUBMED_ID2LABEL.items():
    count = (train_df['label_id'] == idx).sum()
    print(f"  {idx}: {label:<20s} ({count:,} samples)")

## 7. Helper Functions

In [ ]:
def normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / np.maximum(norms, 1e-8)


def extract_keywords_from_text(texts: list, top: int = 200, seed: int = SEED) -> list:
    random.seed(seed)
    np.random.seed(seed)
    parts = []
    for t in tqdm(texts, desc="Building keyword corpus", unit="sent", leave=True):
        parts.append(str(t))
    joined = " ".join(parts)
    tqdm.write(f"  Running YAKE on {len(joined):,} chars → top {top} keywords...")
    extractor = yake.KeywordExtractor(lan="en", n=1, dedupLim=0.9, top=top, features=None)
    keywords = extractor.extract_keywords(joined)
    tqdm.write(f"  YAKE done — {len(keywords)} keywords extracted.")
    return [k[0] for k in keywords]


def embed_texts_with_gpt2model(texts, model, tokenizer, batch_size=32, max_len=MAX_LENGTH):
    model.eval()
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", leave=False):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
            last_hidden = out.last_hidden_state
            seq_lens = enc['attention_mask'].sum(dim=1) - 1
            b_idx = torch.arange(last_hidden.size(0), device=device)
            embs = last_hidden[b_idx, seq_lens]
        all_embs.append(embs.cpu().float().numpy())
    return np.vstack(all_embs)


def embed_texts_with_seq_clf(texts, model, tokenizer, batch_size=32, max_len=MAX_LENGTH):
    model.eval()
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", leave=False):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
            last_hidden = out.hidden_states[-1]
            seq_lens = enc['attention_mask'].sum(dim=1) - 1
            b_idx = torch.arange(last_hidden.size(0), device=device)
            embs = last_hidden[b_idx, seq_lens]
        all_embs.append(embs.cpu().float().numpy())
    return np.vstack(all_embs)


def compute_bee_scores(
    classifier_weights_norm: np.ndarray,
    keyword_embeddings_norm: np.ndarray,
    keywords: list,
    id2label: dict,
):
    """
    BEE SC scores — paper Eq. 2:
        s+_{k,i} = w*ᵀ_k M(c_i)  −  min_{k'} w*ᵀ_{k'} M(c_i)

    Returns
    -------
    bee_df    : DataFrame, one row per keyword, one column per class (s+_{k,i} ≥ 0).
    sorted_sc : (num_classes, num_keywords) SC scores, same sort order as bee_df.
    raw_sims  : (num_classes, num_keywords) raw cosine similarities (unsorted).
    """
    raw_sims  = classifier_weights_norm @ keyword_embeddings_norm.T
    min_sim   = raw_sims.min(axis=0, keepdims=True)
    sc_scores = raw_sims - min_sim                        # (K, N), all ≥ 0

    overall_bias    = sc_scores.max(axis=0)
    order           = np.argsort(overall_bias)[::-1]
    sorted_keywords = np.array(keywords)[order]
    sorted_bias     = overall_bias[order]
    sorted_sc       = sc_scores[:, order]

    bee_df = pd.DataFrame({'Keyword': sorted_keywords, 'Bias_Score': sorted_bias})
    for i, label in id2label.items():
        bee_df[label] = sorted_sc[i]

    return bee_df, sorted_sc, raw_sims


def visualize_bee_results(
    bee_df: pd.DataFrame,
    id2label: dict,
    title_prefix: str = "",
    top_k: int = 15,
):
    """
    One heatmap per class.  Each heatmap shows the top_k keywords ranked
    by s+_{k,i} for that class (rows), with the SC score for that single
    class (one column).  Keywords are sorted highest → lowest.
    """
    for class_idx, class_label in id2label.items():
        # Top-k keywords for this class, sorted by their SC score descending
        top = (
            bee_df[['Keyword', class_label]]
            .nlargest(top_k, class_label)
            .reset_index(drop=True)
        )
        heatmap_data = top.set_index('Keyword')[[class_label]]

        fig, ax = plt.subplots(figsize=(3, max(4, top_k * 0.45)))
        sns.heatmap(
            heatmap_data, cmap="YlOrRd", vmin=0,
            annot=True, fmt=".2f", cbar=True, ax=ax,
        )
        ax.set_title(
            f"{title_prefix}\nClass: {class_label}  —  top {top_k} keywords",
            fontsize=11,
        )
        ax.set_xlabel("")
        ax.set_ylabel("Keyword")
        ax.set_xticklabels([class_label], rotation=0)
        plt.tight_layout()
        plt.show()


print("Helper functions defined.")

## 8. Target Model — `devmanpreet/Medical-GPT2-Classifier`

Loaded as-is. No fine-tuning performed.

In [ ]:
from transformers import GPT2ForSequenceClassification, GPT2Config
from huggingface_hub import hf_hub_download

print("=" * 60)
print(f"Loading TARGET model: {MODEL_TARGET_ID}")
print("=" * 60)

# ------------------------------------------------------------------
# Tokenizer: use base GPT-2 (identical vocab)
# ------------------------------------------------------------------
target_tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_ID)
if target_tokenizer.pad_token is None:
    target_tokenizer.pad_token = target_tokenizer.eos_token
print(f"Tokenizer: {MODEL_BASE_ID}")

# ------------------------------------------------------------------
# Download raw .pth checkpoint (cached after first run)
# ------------------------------------------------------------------
pth_path = hf_hub_download(repo_id=MODEL_TARGET_ID,
                            filename="biofinetuned_partialEpoch1.pth")
ckpt = torch.load(pth_path, map_location="cpu", weights_only=False)
state_dict = ckpt if isinstance(ckpt, dict) else ckpt.state_dict()
print(f"Checkpoint: {len(state_dict)} keys loaded.")

# ------------------------------------------------------------------
# Detect architecture details from actual key names
# ------------------------------------------------------------------
n_layers = max(int(k.split('.')[1])
               for k in state_dict if k.startswith('trf_blocks.')) + 1

ff_second_idx = 2 if any('feedforw.layers.2' in k for k in state_dict) else 1

num_labels      = state_dict['out_head.weight'].shape[0]
print(f"Layers={n_layers}  ff_second_idx={ff_second_idx}  num_labels={num_labels}")

# ------------------------------------------------------------------
# Remap to HuggingFace GPT2ForSequenceClassification format
#
# Custom arch (nn.Linear)  →  HF GPT-2 (Conv1D)
#   weight shape: (out, in)  →  (in, out)  ∴ transpose every weight
#   bias / LayerNorm: no change
#
# Custom LayerNorm stores weight as 'scale_params' and bias as 'shift_params'
# Custom FF module is named 'feedforw' (not 'ff')
# Final norm uses 'scale_params' / 'shift_params' (not 'weight' / 'bias')
# ------------------------------------------------------------------
def remap_to_hf(sd, n_layers, ff_second_idx):
    hf = {}

    # Embeddings
    hf['transformer.wte.weight'] = sd['token_emb.weight']
    hf['transformer.wpe.weight'] = sd['pos_emb.weight']

    for n in range(n_layers):
        p = f'trf_blocks.{n}'

        # Q / K / V → combined c_attn (Conv1D: transpose each Linear weight)
        W_q = sd[f'{p}.attn.W_query.weight']
        W_k = sd[f'{p}.attn.W_key.weight']
        W_v = sd[f'{p}.attn.W_value.weight']
        b_q = sd[f'{p}.attn.W_query.bias']
        b_k = sd[f'{p}.attn.W_key.bias']
        b_v = sd[f'{p}.attn.W_value.bias']
        hf[f'transformer.h.{n}.attn.c_attn.weight'] = torch.cat([W_q.T, W_k.T, W_v.T], dim=1)  # (768, 2304)
        hf[f'transformer.h.{n}.attn.c_attn.bias']   = torch.cat([b_q, b_k, b_v], dim=0)         # (2304,)

        # Output projection
        hf[f'transformer.h.{n}.attn.c_proj.weight'] = sd[f'{p}.attn.out_proj.weight'].T
        hf[f'transformer.h.{n}.attn.c_proj.bias']   = sd[f'{p}.attn.out_proj.bias']

        # Layer norms  ('scale_params' → weight,  'shift_params' → bias)
        hf[f'transformer.h.{n}.ln_1.weight'] = sd[f'{p}.Layernorm1.scale_params']
        hf[f'transformer.h.{n}.ln_1.bias']   = sd[f'{p}.Layernorm1.shift_params']
        hf[f'transformer.h.{n}.ln_2.weight'] = sd[f'{p}.Layernorm2.scale_params']
        hf[f'transformer.h.{n}.ln_2.bias']   = sd[f'{p}.Layernorm2.shift_params']

        # Feed-forward  ('feedforw', not 'ff')
        hf[f'transformer.h.{n}.mlp.c_fc.weight']   = sd[f'{p}.feedforw.layers.0.weight'].T
        hf[f'transformer.h.{n}.mlp.c_fc.bias']     = sd[f'{p}.feedforw.layers.0.bias']
        hf[f'transformer.h.{n}.mlp.c_proj.weight'] = sd[f'{p}.feedforw.layers.{ff_second_idx}.weight'].T
        hf[f'transformer.h.{n}.mlp.c_proj.bias']   = sd[f'{p}.feedforw.layers.{ff_second_idx}.bias']

    # Final layer norm
    hf['transformer.ln_f.weight'] = sd['final_norm.scale_params']
    hf['transformer.ln_f.bias']   = sd['final_norm.shift_params']

    # Classification head (HF score has no bias — skip out_head.bias)
    hf['score.weight'] = sd['out_head.weight']

    return hf

hf_sd = remap_to_hf(state_dict, n_layers, ff_second_idx)
print(f"Remapped {len(hf_sd)} keys.")

# ------------------------------------------------------------------
# Build GPT2ForSequenceClassification shell and load weights
# ------------------------------------------------------------------
cfg = GPT2Config.from_pretrained(MODEL_BASE_ID)
cfg.num_labels   = num_labels
cfg.id2label     = {i: str(i) for i in range(num_labels)}
cfg.label2id     = {str(i): i for i in range(num_labels)}
cfg.pad_token_id = target_tokenizer.pad_token_id

target_model = GPT2ForSequenceClassification(cfg)
missing, unexpected = target_model.load_state_dict(hf_sd, strict=False)

print(f"load_state_dict — missing={len(missing)}  unexpected={len(unexpected)}")
if missing:    print(f"  Missing   : {missing}")
if unexpected: print(f"  Unexpected: {unexpected}")

target_model.to(device)
target_model.eval()

TARGET_CLF_HEAD    = target_model.score
TARGET_ID2LABEL    = target_model.config.id2label
TARGET_LABEL_NAMES = list(TARGET_ID2LABEL.values())

print(f"\nClassifier head : {TARGET_CLF_HEAD.weight.shape}")
print(f"num_labels      : {num_labels}")
print(f"Model has NaN   : {any(torch.isnan(p).any() for p in target_model.parameters())}")
print("Target model ready.")

## 9. Keyword Extraction from PubMed Data

In [ ]:
# Use ALL samples from every split for keyword extraction
sample_texts = train_df[TEXT_COL].tolist()

print(f"Extracting keywords from ALL {len(sample_texts):,} PubMed sentences...")
raw_keywords = extract_keywords_from_text(sample_texts, top=YAKE_TOP_KEYWORDS)

# Filter out class-name words and single characters
filter_words = {'background', 'objective', 'method', 'methods', 'result', 'results',
                'conclusion', 'conclusions', 'abstract'}
clean_keywords = [
    kw for kw in raw_keywords
    if kw.lower() not in filter_words and len(kw.strip()) > 1
]

print(f"Raw keywords  : {len(raw_keywords)}")
print(f"Clean keywords: {len(clean_keywords)}")
print(f"\nTop 20: {clean_keywords[:20]}")

## 10. BEE Analysis — Target Model (Medical-GPT2-Classifier)

In [ ]:
print("Computing BEE scores for TARGET model...")

# 1. Classifier weights
target_weights_raw  = TARGET_CLF_HEAD.weight.detach().cpu().float().numpy()  # (num_classes, H)
target_weights_norm = normalize_embeddings(target_weights_raw)
print(f"Classifier weight shape: {target_weights_raw.shape}")

# 2. Embed keywords using the target model's body
print("Embedding keywords with target model...")
target_kw_embs = embed_texts_with_seq_clf(
    clean_keywords, target_model, target_tokenizer, batch_size=64, max_len=32
)
target_kw_embs_norm = normalize_embeddings(target_kw_embs)
print(f"Keyword embedding shape: {target_kw_embs.shape}")

# 3. BEE SC scores  s+_{k,i} = sim(w_k, M(c_i)) - min_{k'} sim(w_{k'}, M(c_i))
target_bee_df, target_sc_sorted, target_raw_sims = compute_bee_scores(
    target_weights_norm, target_kw_embs_norm, clean_keywords, TARGET_ID2LABEL
)

print(f"\nTop {TOP_K_DISPLAY} BEE keywords — TARGET model:")
print(target_bee_df[['Keyword', 'Bias_Score']].head(TOP_K_DISPLAY).to_string(index=False))

In [ ]:
# ============================================================
# DIAGNOSTIC: Why are the heatmap columns almost identical?
# ============================================================
# Possible causes:
#   (A) Remapping bug  — transposing weights incorrectly produces
#       nearly parallel class weight vectors.
#   (B) Undertrained model — biofinetuned_partialEpoch1.pth was
#       only run for < 1 epoch, so class weights never diverged.
# ============================================================

print("=" * 65)
print("DIAGNOSTIC — Target Model BEE")
print("=" * 65)

# ----------------------------------------------------------
# 1. Pairwise cosine similarity between the 5 class weight
#    vectors.  If these are all ~ 1.0 the classifier head
#    never specialised → undertrained.
#    If they show diversity but BEE is still flat → embedding
#    extraction or similarity computation is the bug.
# ----------------------------------------------------------
pairwise_cls = sklearn_cosine_sim(target_weights_norm)   # (5, 5)
cls_labels   = TARGET_LABEL_NAMES

print("\n[1] Pairwise cosine similarity — TARGET class weight vectors:")
pairwise_df = pd.DataFrame(pairwise_cls,
                            index=cls_labels,
                            columns=cls_labels)
print(pairwise_df.round(4).to_string())

off_diag = pairwise_cls[np.triu_indices(len(cls_labels), k=1)]
print(f"\n  Off-diagonal stats → mean: {off_diag.mean():.4f}  "
      f"min: {off_diag.min():.4f}  max: {off_diag.max():.4f}")
print("  (near 1.0 = class vectors almost identical = undertrained/collapsed head)")

# ----------------------------------------------------------
# 2. BEE bias score statistics
# ----------------------------------------------------------
bs = target_bee_df['Bias_Score']
print(f"\n[2] Bias score stats:")
print(f"  min={bs.min():.6f}  max={bs.max():.6f}  "
      f"mean={bs.mean():.6f}  std={bs.std():.6f}")
print(f"  Non-zero entries: {(bs > 1e-6).sum()} / {len(bs)}")

# ----------------------------------------------------------
# 3. Per-class similarity range across keywords
# ----------------------------------------------------------
print("\n[3] Per-class similarity range over all keywords:")
for col in TARGET_LABEL_NAMES:
    col_vals = target_bee_df[col].values
    print(f"  Class '{col}':  min={col_vals.min():.4f}  "
          f"max={col_vals.max():.4f}  std={col_vals.std():.6f}")

# ----------------------------------------------------------
# 4. Quick inference sanity check — feed one sentence per
#    pubmed class and check if the model discriminates.
# ----------------------------------------------------------
print("\n[4] Inference sanity check (one sample per PubMed class):")
sample_rows = []
for cid in range(PUBMED_NUM_LABELS):
    rows = train_df[train_df['label_id'] == cid]
    if len(rows):
        sample_rows.append(rows.iloc[0][TEXT_COL])

sample_enc = target_tokenizer(
    sample_rows,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors='pt',
).to(device)

with torch.no_grad():
    logits = target_model(**sample_enc).logits.cpu().float()

pred_classes = logits.argmax(dim=-1).tolist()
print(f"  Logits (one row per sample):")
for i, (row, preds) in enumerate(zip(logits.numpy(), pred_classes)):
    true_label = PUBMED_ID2LABEL[i]
    pred_label = TARGET_ID2LABEL[preds]
    print(f"  Sample[{i}] true_pubmed='{true_label}'  "
          f"pred_target='{pred_label}'  logits={np.round(row, 3)}")

logit_std_across_classes = logits.std(dim=-1)
print(f"\n  Logit std per sample (should be > 0.1 for a non-collapsed model):")
for i, s in enumerate(logit_std_across_classes.tolist()):
    print(f"    Sample[{i}]: std = {s:.4f}")

# ----------------------------------------------------------
# 5. Check raw (un-normalised) classifier weight norms
# ----------------------------------------------------------
print("\n[5] L2 norm of each (raw) class weight vector:")
for i, name in enumerate(TARGET_LABEL_NAMES):
    n = float(np.linalg.norm(target_weights_raw[i]))
    print(f"  Class '{name}': L2 norm = {n:.6f}")

# ----------------------------------------------------------
# Summary verdict
# ----------------------------------------------------------
mean_off_diag = float(off_diag.mean())
print("\n" + "=" * 65)
if mean_off_diag > 0.99:
    print("VERDICT: Class weight vectors are nearly IDENTICAL (mean cos-sim "
          f"= {mean_off_diag:.4f}).")
    print("  → Likely cause: model was barely trained (partialEpoch1).")
    print("    The classification head initialised near-uniformly and never")
    print("    diverged enough for BEE to distinguish classes.")
    print("  → The remapping itself is probably correct; the model is the issue.")
elif mean_off_diag > 0.90:
    print(f"VERDICT: Class vectors moderately similar (mean cos-sim = {mean_off_diag:.4f}).")
    print("  → Check [3]: if per-class sim range is very small, the embedding")
    print("    space is collapsed.  Consider verifying the Conv1D transpose.")
else:
    print(f"VERDICT: Class vectors look distinct (mean cos-sim = {mean_off_diag:.4f}).")
    print("  → Bug is likely in BEE computation or embedding extraction,")
    print("    NOT in the weight remapping.")

In [ ]:
# Visualise
visualize_bee_results(
    target_bee_df, TARGET_ID2LABEL,
    title_prefix="Medical-GPT2-Classifier (Target)",
    top_k=TOP_K_DISPLAY,
)

## 11. Reference Model — `openai-community/gpt2` (frozen body + sklearn probe)

We extract embeddings from the **frozen** GPT-2 body, then fit a `LogisticRegression` on those embeddings.  
The LR coefficients serve as the reference classifier weights for BEE — **no PyTorch training** occurs.

In [ ]:
print("=" * 60)
print(f"Loading BASE model body: {MODEL_BASE_ID}")
print("=" * 60)

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_ID)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

# Load only the transformer body (no classification head)
base_gpt2 = GPT2Model.from_pretrained(MODEL_BASE_ID, torch_dtype=torch.float32)
base_gpt2.to(device)
base_gpt2.eval()

# Sanity check
has_nan = any(torch.isnan(p).any() for p in base_gpt2.parameters())
print(f"Base GPT-2 has NaN: {has_nan}")

total_params = sum(p.numel() for p in base_gpt2.parameters())
print(f"Base GPT-2 parameters: {total_params:,} (ALL frozen — no training)")

In [ ]:
# --------------------------------------------------------------------------
# Sample a fixed subset of the training set for the probe (seeded)
# --------------------------------------------------------------------------
rng_probe = np.random.default_rng(SEED)
probe_idx = rng_probe.choice(len(train_df), size=min(N_PROBE_SAMPLES, len(train_df)), replace=False)
probe_df  = train_df.iloc[probe_idx].reset_index(drop=True)

print(f"Probe set size: {len(probe_df):,}")
print(f"Label distribution in probe set:")
print(probe_df[LABEL_COL].value_counts().to_string())

In [ ]:
# --------------------------------------------------------------------------
# Extract frozen embeddings from base GPT-2 for the probe samples
# --------------------------------------------------------------------------
print("Extracting frozen GPT-2 embeddings for probe samples...")

probe_texts  = probe_df[TEXT_COL].tolist()
probe_labels = probe_df['label_id'].to_numpy()

probe_embeddings = embed_texts_with_gpt2model(
    probe_texts, base_gpt2, base_tokenizer, batch_size=64, max_len=MAX_LENGTH
)

print(f"Probe embeddings shape : {probe_embeddings.shape}")
print(f"Probe labels shape     : {probe_labels.shape}")

In [ ]:
# --------------------------------------------------------------------------
# Fit sklearn LogisticRegression on frozen embeddings
# Coefficients shape: (num_classes, hidden_size) — used as BEE classifier weights
# --------------------------------------------------------------------------
print("Fitting LogisticRegression probe on frozen GPT-2 embeddings...")

lr_probe = LogisticRegression(
    random_state=SEED,
    max_iter=LR_MAX_ITER,
    solver='lbfgs',
    multi_class='multinomial',
    C=1.0,
    n_jobs=-1,
)
lr_probe.fit(probe_embeddings, probe_labels)

# Evaluate probe
probe_preds = lr_probe.predict(probe_embeddings)
probe_acc   = accuracy_score(probe_labels, probe_preds)
probe_f1    = f1_score(probe_labels, probe_preds, average='weighted')

print(f"\nProbe train accuracy : {probe_acc:.4f}")
print(f"Probe train F1       : {probe_f1:.4f}")
print(f"\nClassification report:")
print(classification_report(
    probe_labels, probe_preds,
    target_names=[PUBMED_ID2LABEL[i] for i in range(PUBMED_NUM_LABELS)]
))

# Classifier weights from LR: shape (num_classes, hidden_size)
base_weights_raw  = lr_probe.coef_.astype(np.float32)   # (num_classes, H)
base_weights_norm = normalize_embeddings(base_weights_raw)
print(f"\nLR coefficient shape (= classifier weights): {base_weights_raw.shape}")

In [ ]:
# --------------------------------------------------------------------------
# Embed keywords using the frozen base GPT-2 body
# --------------------------------------------------------------------------
print("Embedding keywords with base GPT-2 body...")

base_kw_embs = embed_texts_with_gpt2model(
    clean_keywords, base_gpt2, base_tokenizer, batch_size=64, max_len=32
)
base_kw_embs_norm = normalize_embeddings(base_kw_embs)
print(f"Keyword embedding shape: {base_kw_embs.shape}")

# --------------------------------------------------------------------------
# BEE SC scores  s+_{k,i} = sim(w_k, M(c_i)) - min_{k'} sim(w_{k'}, M(c_i))
# --------------------------------------------------------------------------
base_bee_df, base_sc_sorted, base_raw_sims = compute_bee_scores(
    base_weights_norm, base_kw_embs_norm, clean_keywords, PUBMED_ID2LABEL
)

print(f"\nTop {TOP_K_DISPLAY} BEE keywords — REFERENCE model:")
print(base_bee_df[['Keyword', 'Bias_Score']].head(TOP_K_DISPLAY).to_string(index=False))

In [ ]:
# Visualise
visualize_bee_results(
    base_bee_df, PUBMED_ID2LABEL,
    title_prefix="GPT-2 Base + sklearn Probe (Reference)",
    top_k=TOP_K_DISPLAY,
)

## 13. Full BEE Tables (All Keywords, Alphabetical)

In [ ]:
for bee_df, model_name in [
    (target_bee_df, f"Medical-GPT2-Classifier — classes: {TARGET_LABEL_NAMES}"),
    (base_bee_df,   f"GPT-2 Base + sklearn Probe — classes: {list(PUBMED_ID2LABEL.values())}"),
]:
    sorted_alpha = bee_df.sort_values('Keyword', ascending=True).reset_index(drop=True)
    label_cols = [c for c in sorted_alpha.columns if c not in ('Keyword', 'Bias_Score')]

    print(f"\n{'='*80}")
    print(f"BEE TABLE — {model_name} ({len(sorted_alpha)} keywords, A-Z)")
    print(f"{'='*80}")

    styled = (
        sorted_alpha.style
        .background_gradient(subset=label_cols, cmap="coolwarm", axis=None)
        .background_gradient(subset=["Bias_Score"], cmap="YlOrRd")
        .format({c: "{:.4f}" for c in label_cols + ["Bias_Score"]})
        .set_caption(model_name)
    )
    try:
        from IPython.display import display
        display(styled)
    except Exception:
        print(sorted_alpha.to_string())

## 14. Comparison of BEE Fingerprints

Since the two models may have **different numbers of classes**, we compare them using:
1. **Bias-score vector** — one scalar per keyword, model-agnostic.
2. **Top-K keyword overlap** — Jaccard index.
3. **Scatter plot + histogram** of bias scores.

In [ ]:
# Align both DataFrames on the shared keyword list
all_keywords_sorted = sorted(set(clean_keywords))

def build_bias_vector(bee_df: pd.DataFrame, keyword_list: list) -> np.ndarray:
    """Return bias score for each keyword in keyword_list (0 if missing)."""
    kw2score = dict(zip(bee_df['Keyword'], bee_df['Bias_Score']))
    return np.array([kw2score.get(kw, 0.0) for kw in keyword_list], dtype=np.float32)

target_bias_vec = build_bias_vector(target_bee_df, all_keywords_sorted)
base_bias_vec   = build_bias_vector(base_bee_df,   all_keywords_sorted)

# Cosine similarity between BEE bias vectors
cos_sim = sklearn_cosine_sim(
    target_bias_vec.reshape(1, -1),
    base_bias_vec.reshape(1, -1),
)[0, 0]

# Pearson correlation
pearson_r = float(np.corrcoef(target_bias_vec, base_bias_vec)[0, 1])

print("=" * 60)
print("BEE FINGERPRINT COMPARISON")
print("=" * 60)
print(f"  Cosine similarity (bias vectors) : {cos_sim:.4f}")
print(f"  Pearson correlation              : {pearson_r:.4f}")
print()
print("Interpretation:")
print("  High similarity → Medical-GPT2 bias fingerprint matches PubMed probe")
print("  Low similarity  → Models biased toward different vocabularies")

In [ ]:
# --------------------------------------------------------------------------
# Build aligned per-class similarity arrays for both models.
#
# Problem: target_bee_df and base_bee_df are each sorted by their own
# bias score (descending), so their rows are in different keyword orders.
# Comparing them column-by-column would silently mix up keywords.
#
# Solution: re-index every column onto all_keywords_sorted (alphabetical,
# shared between both models) and store everything in fingerprint_store.
# Missing keywords are filled with 0.0.
# --------------------------------------------------------------------------

def build_class_vector(bee_df: pd.DataFrame, keyword_list: list, class_col: str) -> np.ndarray:
    """Return per-class similarity for each keyword in keyword_list (0 if missing)."""
    kw2sim = dict(zip(bee_df['Keyword'], bee_df[class_col]))
    return np.array([kw2sim.get(kw, 0.0) for kw in keyword_list], dtype=np.float32)


# -- Target model: one array per class, shape (N_keywords,) --
target_class_cols = [c for c in target_bee_df.columns if c not in ('Keyword', 'Bias_Score')]
target_aligned_sims = {
    col: build_class_vector(target_bee_df, all_keywords_sorted, col)
    for col in target_class_cols
}

# -- Reference model: one array per class, shape (N_keywords,) --
base_class_cols = [c for c in base_bee_df.columns if c not in ('Keyword', 'Bias_Score')]
base_aligned_sims = {
    col: build_class_vector(base_bee_df, all_keywords_sorted, col)
    for col in base_class_cols
}

# --------------------------------------------------------------------------
# fingerprint_store — the single source of truth for all future metrics.
#
# Layout
# ------
# fingerprint_store
# ├── 'keywords'          list[str]           shared keyword index (alphabetical)
# ├── 'target'
# │   ├── 'bias_vec'      np.array (N,)       max-min gap per keyword
# │   ├── 'class_sims'    dict[str → (N,)]    per-class cosine similarity per keyword
# │   ├── 'weights_norm'  np.array (C_t, H)   normalised classifier weights
# │   └── 'kw_embs_norm'  np.array (N, H)     normalised keyword embeddings
# └── 'reference'
#     ├── 'bias_vec'      np.array (N,)
#     ├── 'class_sims'    dict[str → (N,)]
#     ├── 'weights_norm'  np.array (C_r, H)
#     └── 'kw_embs_norm'  np.array (N, H)
# --------------------------------------------------------------------------

# Re-index keyword embeddings to match all_keywords_sorted
# (clean_keywords is in YAKE order; all_keywords_sorted is alphabetical)
_kw_to_idx = {kw: i for i, kw in enumerate(clean_keywords)}

target_kw_embs_aligned = np.array([
    target_kw_embs_norm[_kw_to_idx[kw]] if kw in _kw_to_idx else np.zeros(target_kw_embs_norm.shape[1])
    for kw in all_keywords_sorted
], dtype=np.float32)

base_kw_embs_aligned = np.array([
    base_kw_embs_norm[_kw_to_idx[kw]] if kw in _kw_to_idx else np.zeros(base_kw_embs_norm.shape[1])
    for kw in all_keywords_sorted
], dtype=np.float32)

fingerprint_store = {
    'keywords': all_keywords_sorted,          # shared index — same order for every array below
    'target': {
        'bias_vec':     target_bias_vec,       # (N,)   — aligned on keywords
        'class_sims':   target_aligned_sims,   # {class_name: (N,)}
        'weights_norm': target_weights_norm,   # (C_t, H)
        'kw_embs_norm': target_kw_embs_aligned,# (N, H) — aligned on keywords
    },
    'reference': {
        'bias_vec':     base_bias_vec,         # (N,)   — aligned on keywords
        'class_sims':   base_aligned_sims,     # {class_name: (N,)}
        'weights_norm': base_weights_norm,     # (C_r, H)
        'kw_embs_norm': base_kw_embs_aligned,  # (N, H) — aligned on keywords
    },
}

# --------------------------------------------------------------------------
# Quick sanity check
# --------------------------------------------------------------------------
N = len(all_keywords_sorted)
print("fingerprint_store — shapes:")
print(f"  keywords              : {N} entries")
print(f"  target  bias_vec      : {fingerprint_store['target']['bias_vec'].shape}")
print(f"  target  class_sims    : {list(fingerprint_store['target']['class_sims'].keys())}")
print(f"  target  weights_norm  : {fingerprint_store['target']['weights_norm'].shape}")
print(f"  target  kw_embs_norm  : {fingerprint_store['target']['kw_embs_norm'].shape}")
print(f"  reference bias_vec    : {fingerprint_store['reference']['bias_vec'].shape}")
print(f"  reference class_sims  : {list(fingerprint_store['reference']['class_sims'].keys())}")
print(f"  reference weights_norm: {fingerprint_store['reference']['weights_norm'].shape}")
print(f"  reference kw_embs_norm: {fingerprint_store['reference']['kw_embs_norm'].shape}")
print()
print("All arrays share the same keyword index — safe to compare element-wise.")
print()
print("Example — add any metric below, e.g.:")
print("  from scipy.stats import spearmanr, kendalltau")
print("  spearmanr(fingerprint_store['target']['bias_vec'],")
print("            fingerprint_store['reference']['bias_vec'])")

### Central Fingerprint Store
All BEE data aligned on `all_keywords_sorted` and collected into `fingerprint_store`.  
Add any future metric in a new cell below — every array you need is already here.

In [ ]:
# --------------------------------------------------------------------------
# Top-K keyword overlap (Jaccard index)
# --------------------------------------------------------------------------
target_top_k = set(target_bee_df['Keyword'].head(TOP_K_OVERLAP).tolist())
base_top_k   = set(base_bee_df['Keyword'].head(TOP_K_OVERLAP).tolist())

overlap = target_top_k & base_top_k
jaccard = len(overlap) / len(target_top_k | base_top_k)

print(f"Top-{TOP_K_OVERLAP} keyword overlap:")
print(f"  Target model top-{TOP_K_OVERLAP}   : {sorted(target_top_k)}")
print(f"  Reference model top-{TOP_K_OVERLAP}: {sorted(base_top_k)}")
print(f"  Shared keywords ({len(overlap)})   : {sorted(overlap)}")
print(f"  Jaccard index            : {jaccard:.4f}")

## 15. Visualisations — Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Plot 1: Bias score distributions ---
axes[0].hist(target_bias_vec, bins=30, alpha=0.6, label='Medical-GPT2 (Target)', color='tomato', edgecolor='white')
axes[0].hist(base_bias_vec,   bins=30, alpha=0.6, label='GPT-2 Base + Probe (Ref)', color='steelblue', edgecolor='white')
axes[0].set_title("BEE Bias Score Distribution", fontsize=12)
axes[0].set_xlabel("Bias Score (Max − Min sim)")
axes[0].set_ylabel("Count")
axes[0].legend()

# --- Plot 2: Scatter — target vs reference bias scores ---
axes[1].scatter(base_bias_vec, target_bias_vec, alpha=0.4, s=18, color='purple')
axes[1].set_xlabel("Reference (GPT-2 + Probe) Bias Score")
axes[1].set_ylabel("Target (Medical-GPT2) Bias Score")
axes[1].set_title(f"Bias Score Scatter\n(Pearson r = {pearson_r:.4f}, cos_sim = {cos_sim:.4f})", fontsize=11)

# Add diagonal reference line
lims = [min(base_bias_vec.min(), target_bias_vec.min()),
        max(base_bias_vec.max(), target_bias_vec.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1, alpha=0.5, label='y = x')
axes[1].legend()

# --- Plot 3: Side-by-side bar chart for top-K ---
top_common = target_bee_df.head(TOP_K_OVERLAP).copy()
base_score_map = dict(zip(base_bee_df['Keyword'], base_bee_df['Bias_Score']))
top_common['Ref_Score'] = top_common['Keyword'].map(lambda k: base_score_map.get(k, 0.0))

x     = np.arange(len(top_common))
width = 0.4
axes[2].barh(x - width/2, top_common['Bias_Score'], width,
             label='Medical-GPT2', color='tomato', alpha=0.85)
axes[2].barh(x + width/2, top_common['Ref_Score'],  width,
             label='GPT-2 + Probe', color='steelblue', alpha=0.85)
axes[2].set_yticks(x)
axes[2].set_yticklabels(top_common['Keyword'], fontsize=9)
axes[2].set_title(f"Top-{TOP_K_OVERLAP} Keywords by Target BEE Score", fontsize=11)
axes[2].set_xlabel("Bias Score")
axes[2].legend()

plt.suptitle("BEE Analysis — Medical-GPT2 vs GPT-2 Base Comparison",
             fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Bias-score heatmap for top-N shared & exclusive keywords
# --------------------------------------------------------------------------
N_HEATMAP = 25

# Combine and sort by target bias score
compare_df = pd.DataFrame({'Keyword': all_keywords_sorted,
                            'Target': target_bias_vec,
                            'Reference': base_bias_vec})
compare_df = compare_df.sort_values('Target', ascending=False).head(N_HEATMAP)

fig, ax = plt.subplots(figsize=(6, max(4, N_HEATMAP // 2)))
heat = compare_df.set_index('Keyword')[['Target', 'Reference']]
sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlOrRd",
            cbar_kws={'label': 'BEE Bias Score'}, ax=ax)
ax.set_title(f"Top-{N_HEATMAP} BEE Keywords by Target Score", fontsize=13)
ax.set_xlabel("Model")
ax.set_ylabel("Keyword")
plt.tight_layout()
plt.show()

## 16. Export Results to Excel

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

OUTPUT_EXCEL = "bee_results_gpt2_pubmed.xlsx"

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

    # Sheet 1 — Target model BEE (sorted by bias score)
    target_bee_df.to_excel(writer, sheet_name='Target_Medical-GPT2', index=False)

    # Sheet 2 — Reference model BEE (sorted by bias score)
    base_bee_df.to_excel(writer, sheet_name='Ref_GPT2-Probe', index=False)

    # Sheet 3 — Side-by-side comparison
    compare_all = pd.DataFrame({
        'Keyword': all_keywords_sorted,
        'Target_BiasScore': target_bias_vec,
        'Ref_BiasScore':    base_bias_vec,
        'Delta':            target_bias_vec - base_bias_vec,
    }).sort_values('Target_BiasScore', ascending=False).reset_index(drop=True)
    compare_all.to_excel(writer, sheet_name='Comparison', index=False)

    # Sheet 4 — Summary metrics
    summary_data = {
        'Metric': [
            'Dataset',
            'Target model',
            'Target num_labels',
            'Target label names',
            'Reference model',
            'Reference num_labels',
            'Reference label names',
            'Num keywords',
            'Cosine similarity (bias vectors)',
            'Pearson correlation (bias vectors)',
            f'Jaccard top-{TOP_K_OVERLAP}',
            'SEED',
        ],
        'Value': [
            DATASET_NAME,
            MODEL_TARGET_ID,
            target_model.config.num_labels,
            str(TARGET_LABEL_NAMES),
            MODEL_BASE_ID,
            PUBMED_NUM_LABELS,
            str(list(PUBMED_ID2LABEL.values())),
            len(clean_keywords),
            f"{cos_sim:.4f}",
            f"{pearson_r:.4f}",
            f"{jaccard:.4f}",
            SEED,
        ],
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

print(f"Results exported to: {OUTPUT_EXCEL}")

## 17. Final Summary

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"""
Dataset : {DATASET_NAME}
  Train samples used : {len(train_df):,}
  Classes            : {PUBMED_NUM_LABELS} → {list(PUBMED_ID2LABEL.values())}

TARGET model : {MODEL_TARGET_ID}
  Classes     : {target_model.config.num_labels} → {TARGET_LABEL_NAMES}
  Training    : none (used as-is)

REFERENCE model : {MODEL_BASE_ID}
  Body        : fully frozen GPT-2
  Probe       : sklearn LogisticRegression on {N_PROBE_SAMPLES:,} frozen embeddings
  Classes     : {PUBMED_NUM_LABELS} → {list(PUBMED_ID2LABEL.values())}
  Training    : none (sklearn fit on frozen features, no PyTorch gradients)

BEE Analysis
  Keywords extracted (YAKE)          : {len(clean_keywords)}
  Cosine similarity (bias vectors)   : {cos_sim:.4f}
  Pearson correlation (bias vectors) : {pearson_r:.4f}
  Jaccard top-{TOP_K_OVERLAP} keywords               : {jaccard:.4f}

Reproducibility
  SEED = {SEED}  (set for random, numpy, torch, cuda, sklearn, YAKE)
""")

print("=" * 70)

---
### Interpretation Guide

| Metric | High value | Low value |
|---|---|---|
| **Cosine similarity** | Target fingerprint matches PubMed probe | Different biased vocabulary |
| **Pearson r** | Strong linear agreement in keyword biases | Uncorrelated biases |
| **Jaccard top-K** | Same keywords are most biased in both models | Different top keywords |